# Engine v4 — Stockfish (NNUE)

Stockfish 16 carries two evaluations: the NNUE network (the default) and the hand-written classical one; the UCI option `Use NNUE` switches between them. Stockfish 16.1 and later removed the classical evaluation, so this needs Stockfish 16 (`stockfish16/`, see RECIPE.md).

Everything runs from `engines/v5-stockfish/`: the harness is `scripts/play_stockfish.py`, the measurements `scripts/run_experiments.py`.

In [1]:
import os, sys
if not os.path.isdir("scripts") and os.path.isdir(os.path.join("engines", "v5-stockfish")):
    os.chdir(os.path.join("engines", "v5-stockfish"))       # opened from the portfolio root
sys.path.insert(0, "scripts")

import chess
from play_stockfish import find_stockfish, open_stockfish, play_stockfish, moves_from_pgn, summarize, same_move
import run_experiments as rx

eng = open_stockfish(True)
print(eng.id["name"], "| Use NNUE option:", eng.options["Use NNUE"])
eng.quit()

Stockfish 16 | Use NNUE option: Option(name='Use NNUE', type='check', default=True, min=None, max=None, var=[])


## Task 1 — what the first layer stores, and what a move changes

HalfKP: one input for every (my king square, piece square, piece type and colour) for the nine non-king piece types of both colours, seen from each side. The first layer's output, the *accumulator*, is just the sum of the weight rows of the inputs that are on. So a move only has to add and subtract the rows of the inputs it turns on and off. The cell below counts those rows for a quiet move, a capture and a king move, for each side's half of the accumulator.

In [2]:
def halfkp(board, side):
    """The active HalfKP features from one side's point of view: (king square, piece square, piece)."""
    k = board.king(side)
    return {(k, sq, p.symbol()) for sq, p in board.piece_map().items() if p.piece_type != chess.KING}

def rows_touched(fen, san):
    b = chess.Board(fen)
    before = {s: halfkp(b, s) for s in (chess.WHITE, chess.BLACK)}
    b.push_san(san)
    out = {}
    for s, name in ((chess.WHITE, "White half"), (chess.BLACK, "Black half")):
        after = halfkp(b, s)
        out[name] = (len(before[s] - after), len(after - before[s]), len(after))
    return out

italian = "r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/3P1N2/PPP2PPP/RNBQK2R w KQkq - 1 5"
for label, fen, san in [("quiet move", italian, "Nc3"),
                        ("capture", "r1bqkbnr/pppp1ppp/2n5/4p3/3PP3/5N2/PPP2PPP/RNBQKB1R b KQkq - 0 3", "exd4"),
                        ("king move", italian, "Kf1")]:
    for half, (off, on, total) in rows_touched(fen, san).items():
        print(f"{label:<11} {san:<5} {half}: {off} rows subtracted, {on} added   (of {total} active)")

quiet move  Nc3   White half: 1 rows subtracted, 1 added   (of 30 active)
quiet move  Nc3   Black half: 1 rows subtracted, 1 added   (of 30 active)
capture     exd4  White half: 2 rows subtracted, 1 added   (of 29 active)
capture     exd4  Black half: 2 rows subtracted, 1 added   (of 29 active)
king move   Kf1   White half: 30 rows subtracted, 30 added   (of 30 active)
king move   Kf1   Black half: 0 rows subtracted, 0 added   (of 30 active)


## Task 2.1 — a full game against me, every move with depth, nodes, nodes per second and score

Stockfish is White at depth 10. The two games were played move by move (`python scripts/play_stockfish.py --opponent you --nnue on` / `off`). Here they are replayed from the saved PGNs: Black's moves come from the file, Stockfish searches every one of its own moves again, so the numbers printed are live. (One thread, a fresh hash per game, a fixed depth: the replay comes out move for move the same.)

In [3]:
board_on, s_on = play_stockfish("you", nnue=True, depth=10,
                                moves=moves_from_pgn("games/v4-stockfish-nnue-vs-me.pgn"),
                                out="games/v4-stockfish-nnue-vs-me.pgn")

engine.id = {'name': 'Stockfish 16', 'author': 'the Stockfish developers (see AUTHORS file)'} | Use NNUE = True
White: Stockfish 16 (NNUE, depth 10)    Black: me

    1. SF  e4       ( 0.0s)  depth 10      7,548 nodes  1,258,000 n/s  score +50
       opp e5       ( 0.0s)
    2. SF  Nf3      ( 0.0s)  depth 10      2,325 nodes  1,162,500 n/s  score +50
       opp Nc6      ( 0.0s)


    3. SF  Bc4      ( 0.0s)  depth 10     12,601 nodes  1,260,100 n/s  score +57
       opp Bc5      ( 0.0s)
    4. SF  c3       ( 0.0s)  depth 10      5,032 nodes  1,258,000 n/s  score +64
       opp Nf6      ( 0.0s)
    5. SF  d4       ( 0.0s)  depth 10     10,803 nodes  1,200,333 n/s  score +32
       opp exd4     ( 0.0s)
    6. SF  cxd4     ( 0.0s)  depth 10      2,432 nodes  1,216,000 n/s  score +69
       opp Bb4+     ( 0.0s)


    7. SF  Nbd2     ( 0.0s)  depth 10     20,400 nodes  1,275,000 n/s  score +23
       opp d5       ( 0.0s)
    8. SF  exd5     ( 0.0s)  depth 10      5,089 nodes  1,272,250 n/s  score +30
       opp Nxd5     ( 0.0s)
    9. SF  a3       ( 0.0s)  depth 10     14,059 nodes  1,278,090 n/s  score +15
       opp Bxd2+    ( 0.0s)
   10. SF  Bxd2     ( 0.0s)  depth 10      6,088 nodes  1,217,600 n/s  score +10
       opp O-O      ( 0.0s)
   11. SF  O-O      ( 0.0s)  depth 10      4,558 nodes  1,139,500 n/s  score +9
       opp Bg4      ( 0.0s)
   12. SF  h3       ( 0.0s)  depth 10      4,210 nodes  1,403,333 n/s  score +32
       opp Bh5      ( 0.0s)
   13. SF  g4       ( 0.0s)  depth 10      4,075 nodes  1,358,333 n/s  score +70
       opp Bg6      ( 0.0s)
   14. SF  Re1      ( 0.0s)  depth 10     11,998 nodes  1,333,111 n/s  score +75
       opp Nb6      ( 0.0s)
   15. SF  Ba2      ( 0.0s)  depth 10      9,198 nodes  1,149,750 n/s  score +59
       opp Qf6      ( 0.0s)
   16. SF  Bg5      

## Task 2.2 — the same opponent with the network off, same depth

In [4]:
board_off, s_off = play_stockfish("you", nnue=False, depth=10,
                                  moves=moves_from_pgn("games/v4-stockfish-classical-vs-me.pgn"),
                                  out="games/v4-stockfish-classical-vs-me.pgn")

engine.id = {'name': 'Stockfish 16', 'author': 'the Stockfish developers (see AUTHORS file)'} | Use NNUE = False
White: Stockfish 16 (classical, depth 10)    Black: me

    1. SF  e4       ( 0.0s)  depth 10     23,921 nodes  2,174,636 n/s  score +17
       opp e5       ( 0.0s)
    2. SF  d4       ( 0.0s)  depth 10     24,325 nodes  2,027,083 n/s  score +17
       opp exd4     ( 0.0s)
    3. SF  Qxd4     ( 0.0s)  depth 10     65,301 nodes  1,978,818 n/s  score -4
       opp Nc6      ( 0.0s)
    4. SF  Qe3      ( 0.0s)  depth 10     48,730 nodes  2,215,000 n/s  score -4
       opp Nf6      ( 0.0s)
    5. SF  Nd2      ( 0.0s)  depth 10     31,968 nodes  2,131,200 n/s  score -13
       opp Bb4      ( 0.0s)
    6. SF  c3       ( 0.0s)  depth 10     65,572 nodes  2,115,225 n/s  score -16
       opp Ba5      ( 0.0s)
    7. SF  b4       ( 0.0s)  depth 10     62,484 nodes  2,231,571 n/s  score -26
       opp Bb6      ( 0.0s)
    8. SF  Qf4      ( 0.0s)  depth 10     39,390 nodes  1,969,500 n/s 

    9. SF  Bb5      ( 0.0s)  depth 10     92,967 nodes  2,112,886 n/s  score -108
       opp Re8      ( 0.0s)
   10. SF  Ne2      ( 0.0s)  depth 10     18,735 nodes  1,873,500 n/s  score -95
       opp d5       ( 0.0s)
   11. SF  O-O      ( 0.0s)  depth 10     13,142 nodes  1,642,750 n/s  score -97
       opp dxe4     ( 0.0s)
   12. SF  Nc4      ( 0.0s)  depth 10     14,639 nodes  1,626,555 n/s  score -43
       opp Nd5      ( 0.0s)
   13. SF  Qg3      ( 0.0s)  depth 10     22,990 nodes  1,915,833 n/s  score -22
       opp Be6      ( 0.0s)
   14. SF  a4       ( 0.0s)  depth 10     54,765 nodes  2,106,346 n/s  score -4
       opp a6       ( 0.0s)
   15. SF  Bxc6     ( 0.0s)  depth 10     15,837 nodes  1,759,666 n/s  score +8
       opp bxc6     ( 0.0s)
   16. SF  a5       ( 0.0s)  depth 10     48,438 nodes  2,106,000 n/s  score -27
       opp Ba7      ( 0.0s)
   17. SF  Bh6      ( 0.0s)  depth 10      4,747 nodes  1,186,750 n/s  score -29
       opp Qf6      ( 0.0s)
   18. SF  Bg5      

   32. SF  Rxe8     ( 0.0s)  depth 10      5,225 nodes  1,741,666 n/s  score +135
       opp Kxe8     ( 0.0s)
   33. SF  Nc3      ( 0.0s)  depth 10      3,090 nodes  1,030,000 n/s  score +122
       opp Bb3      ( 0.0s)
   34. SF  Ne4      ( 0.0s)  depth 10      9,640 nodes  1,928,000 n/s  score +85
       opp Ke7      ( 0.0s)
   35. SF  Nd2      ( 0.0s)  depth 10     11,315 nodes  2,828,750 n/s  score +102
       opp Bc2      ( 0.0s)
   36. SF  c5       ( 0.0s)  depth 10      7,381 nodes  2,460,333 n/s  score +102
       opp Ke6      ( 0.0s)
   37. SF  e4       ( 0.0s)  depth 10      5,288 nodes  1,762,666 n/s  score +103
       opp Ke5      ( 0.0s)
   38. SF  Ke3      ( 0.0s)  depth 10      3,732 nodes  1,244,000 n/s  score +107
       opp f5       ( 0.0s)
   39. SF  Nf3+     ( 0.0s)  depth 10      3,923 nodes  1,307,666 n/s  score +149
       opp Kf6      ( 0.0s)
   40. SF  e5+      ( 0.0s)  depth 10      3,248 nodes  1,624,000 n/s  score +154
       opp Ke6      ( 0.0s)
   41. SF  

Side by side, and the move check: every position Stockfish faced in each game, searched again with the *other* evaluation at the same depth — how often does it pick the same move?

In [5]:
print(f"{'White':<10} {'result':<7} {'SF moves':>8} {'nodes/move':>11} {'nodes/s':>11} {'same move with other eval':>27}")
for name, nnue, b, s in [("NNUE", True, board_on, s_on), ("classical", False, board_off, s_off)]:
    per_move, nps = summarize(s)
    same, n = same_move(s, nnue, depth=10)
    print(f"{name:<10} {b.result(claim_draw=True):<7} {len(s):>8} {per_move:>11,.0f} {nps:>11,.0f} {f'{same}/{n} ({same / n:.0%})':>27}")

White      result  SF moves  nodes/move     nodes/s   same move with other eval


NNUE       1-0           32       4,742   1,015,421                 17/32 (53%)


classical  1-0           57      16,591   1,756,137                 35/57 (61%)


## Task 3 — the change: the network on → off at the same depth

Everything else identical: Stockfish 16, one thread, 64 MB hash, depth 10, an empty hash before every search. The speed table is six positions; the match is NNUE against classical, 20 games, colours alternating, ten openings each played once with each colour. Both go to `results.csv`. (About ten minutes.)

In [6]:
speed = rx.speed_table(depth=10)

position      eval      depth      nodes    nodes/s      ms  move    score
start         NNUE         10      7,548  1,078,285      22  e4        +50
start         classical    10     23,921  1,993,416      24  e4        +17
Italian       NNUE         10      9,052  1,293,142      14  O-O       +41
Italian       classical    10     40,458  2,129,368      24  O-O        +0
open middle   NNUE         10     25,714  1,285,700      26  h3        +32
open middle   classical    10     57,379  2,125,148      32  a3        +17
Sicilian      NNUE         10     17,303  1,441,916      18  Bg5       +60


Sicilian      classical    10     64,053  1,883,911      39  Bb5       +31
rook endgame  NNUE         10      9,932  1,241,500      14  Rc5        +0
rook endgame  classical    10      5,240  1,746,666       9  f4         +0
king hunt     NNUE         10      8,901  1,271,571      14  c5        +38
king hunt     classical    10     32,994  1,940,823      24  c5        +22


In [7]:
m = rx.match(games=20, depth=10)

  game  1  NNUE      White  1-0      163 plies  NNUE 1.0   (e4 e5 Nf3 Nc6 Bb5 a6)


  game  2  classical White  0-1      194 plies  NNUE 1.0   (e4 e5 Nf3 Nc6 Bb5 a6)


  game  3  NNUE      White  1/2-1/2  233 plies  NNUE 0.5   (e4 e5 Nf3 Nc6 Bc4 Bc5)


  game  4  classical White  0-1      116 plies  NNUE 1.0   (e4 e5 Nf3 Nc6 Bc4 Bc5)


  game  5  NNUE      White  1-0      163 plies  NNUE 1.0   (e4 c5 Nf3 d6 d4 cxd4)


  game  6  classical White  1/2-1/2  184 plies  NNUE 0.5   (e4 c5 Nf3 d6 d4 cxd4)


... (10 outputs left out)

  game 17  NNUE      White  1-0      111 plies  NNUE 1.0   (c4 e5 Nc3 Nf6 g3 d5)


  game 18  classical White  1-0      141 plies  NNUE 0.0   (c4 e5 Nc3 Nf6 g3 d5)


  game 19  NNUE      White  1-0      155 plies  NNUE 1.0   (Nf3 d5 g3 Nf6 Bg2 c6)


  game 20  classical White  0-1      198 plies  NNUE 1.0   (Nf3 d5 g3 Nf6 Bg2 c6)

NNUE 13.0 -- classical 7.0   (14 of 20 decisive)


In [8]:
import csv
rows = speed + [m]
for r in rows:
    r["setting"] = "depth 10"
fields = ["experiment", "setting", "label", "eval", "depth", "nodes", "nps", "time_ms", "move", "score",
          "games", "nnue_points", "classical_points", "decisive"]
with open("results.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print(open("results.csv", encoding="utf-8").read())

experiment,setting,label,eval,depth,nodes,nps,time_ms,move,score,games,nnue_points,classical_points,decisive
speed,depth 10,start,NNUE,10,7548,1078285,23,e4,50,,,,
speed,depth 10,start,classical,10,23921,1993416,24,e4,17,,,,
speed,depth 10,Italian,NNUE,10,9052,1293142,15,O-O,41,,,,
speed,depth 10,Italian,classical,10,40458,2129368,25,O-O,0,,,,
speed,depth 10,open middle,NNUE,10,25714,1285700,27,h3,32,,,,
speed,depth 10,open middle,classical,10,57379,2125148,33,a3,17,,,,
speed,depth 10,Sicilian,NNUE,10,17303,1441916,18,Bg5,60,,,,
speed,depth 10,Sicilian,classical,10,64053,1883911,40,Bb5,31,,,,
speed,depth 10,rook endgame,NNUE,10,9932,1241500,15,Rc5,0,,,,
speed,depth 10,rook endgame,classical,10,5240,1746666,10,f4,0,,,,
speed,depth 10,king hunt,NNUE,10,8901,1271571,14,c5,38,,,,
speed,depth 10,king hunt,classical,10,32994,1940823,25,c5,22,,,,
match,depth 10,"NNUE vs classical, colours alternating, 10 openings x 2",,,,,,,,20,13.0,7.0,14



In [9]:
on = [r for r in speed if r["eval"] == "NNUE"]
off = [r for r in speed if r["eval"] == "classical"]
n_on, n_off = sum(r["nodes"] for r in on), sum(r["nodes"] for r in off)
nps_on, nps_off = sum(r["nps"] for r in on) / len(on), sum(r["nps"] for r in off) / len(off)
print(f"nodes, six positions:    NNUE {n_on:,}   classical {n_off:,}   ({n_off / n_on:.1f}x)")
print(f"nodes per second (mean): NNUE {nps_on:,.0f}   classical {nps_off:,.0f}   ({nps_off / nps_on:.1f}x)")
print(f"same move in {sum(a['move'] == b['move'] for a, b in zip(on, off))} of {len(on)} positions")
print(f"match: NNUE {m['nnue_points']} - classical {m['classical_points']}  ({m['decisive']} of {m['games']} decisive)")

nodes, six positions:    NNUE 78,450   classical 224,045   (2.9x)
nodes per second (mean): NNUE 1,268,686   classical 1,969,889   (1.6x)
same move in 3 of 6 positions
match: NNUE 13.0 - classical 7.0  (14 of 20 decisive)
